# Minimal Typeflux Inline + Anthropic Demo

This notebook is fully self-contained. It shows the smallest useful Typeflux setup:
a typed input model, a typed output model, one workflow step, one local text prompt,
an `InlineResolver`, and one Anthropic provider call.


## What You Are Learning

This tutorial focuses on the core Typeflux mental model:

- **Pydantic models** define the workflow contract.
- A **step** declares which prompt to use and what output schema to validate.
- A **workflow** is just an ordered collection of steps.
- An **InlineResolver** supplies prompt text from an in-memory map.
- A **provider** turns rendered messages into validated structured output.

The example stays intentionally small so each piece is easy to see.


## Prerequisite

Install this tutorial project directly inside `tutorial/` so the example is self-contained:

```bash
cd /Users/josephgibli/GitHub/typeflux-test/tutorial
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ -e .
```

Then make sure `ANTHROPIC_API_KEY` is available to the notebook kernel before running the live cells.


## Imports and Fundamentals

`typeflux` keeps the primitive pieces very small:

- `PromptRef` names the prompt a step wants.
- `ResolvedPrompt` holds the actual prompt text after you load it.
- `InlineResolver` maps prompt names to prompt bodies.
- `@step(...)` turns a typed function into a `StepSpec`.
- `WorkflowSpec` defines the ordered execution plan.
- `run_workflow(...)` executes the whole pipeline.

For this notebook, we use exactly one step so the control flow is easy to follow.


In [42]:
import os
from pathlib import Path

from pydantic import BaseModel, Field
from typeflux import InlineResolver, PromptRef, ResolvedPrompt, WorkflowSpec, run_workflow, step
from typeflux.providers.anthropic import AnthropicProvider


## Step Contracts

Typeflux uses Pydantic models as the source of truth for workflow data.

- `TopicInput` is what our workflow starts with.
- `Explanation` is what we want the model to return.
- The `@step` decorator links the Python step to a prompt name.
- The step function can also contain post-LLM business logic.

In this minimal example, the hook simply returns the validated model output unchanged.


In [43]:
class TopicInput(BaseModel):
    topic: str = Field(description="Topic to explain in one sentence.")

class Explanation(BaseModel):
    title: str = Field(description="Short title for the explanation.")
    explanation: str = Field(description="One concise sentence that teaches the idea.")

@step(prompt=PromptRef("tutorial-explain-topic"))
def explain_topic(input: TopicInput, output: Explanation) -> Explanation:
    # In a larger app, this is where you could add logging,
    # guardrails, deterministic business rules, or cleanup logic.
    return output


workflow = WorkflowSpec(
    name="InlineAnthropicTutorial",
    steps=(explain_topic,),
)


## Prompt Resolution

The prompt lives in a local text file instead of being embedded directly in Python or fetched from a remote registry.

A few important ideas:

- The prompt text contains `{{topic}}`, which Typeflux renders from `TopicInput`.
- `PromptRef("tutorial-explain-topic")` is the key the step asks for.
- We read the file ourselves, then wrap that text with `ResolvedPrompt.from_text(...)`.
- `InlineResolver` provides the `ResolvedPrompt` for that key.
- `template_format="mustache"` tells Typeflux how to render placeholders.

This is a simple pattern when you want prompt files on disk without needing a remote prompt registry.


In [ ]:
%env ANTHROPIC_API_KEY="[INSERT HERE]"

In [45]:
prompt_path = Path("prompts/explain.txt")

prompt_ref = PromptRef("tutorial-explain-topic", version="production")
prompt_text = prompt_path.read_text(encoding="utf-8")

resolver = InlineResolver(
    prompts={
        (prompt_ref.name, prompt_ref.version): ResolvedPrompt.from_text(
            prompt_ref,
            prompt_text,
            template_format="mustache",
        )
    }
)

api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise RuntimeError("Set ANTHROPIC_API_KEY before running the live demo cells.")

provider = AnthropicProvider(
    api_key=api_key,
    default_model=os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-4-5"),
)


## Execution Flow

When `run_workflow(...)` executes this example, Typeflux does five conceptual things:

1. Validates that the initial object is a `TopicInput`.
2. Reads `prompts/explain.txt` into a `ResolvedPrompt`.
3. Fetches that prompt from the resolver and renders `{{topic}}`.
4. Calls Anthropic and asks for a response matching the `Explanation` schema.
5. Returns a validated `Explanation` object back to Python.

That validated object is the main payoff of the framework: you work with typed data,
not ad hoc JSON parsing.


In [ ]:
result = run_workflow(
    workflow,
    TopicInput(topic="vector databases"),
    resolver=resolver,
    provider=provider,
)

result


## Why This Example Matters

Even though this workflow has only one step, it demonstrates the same primitives you use in larger systems:

- add more `BaseModel` schemas as the workflow grows
- add more `@step` functions for multi-stage pipelines
- swap `InlineResolver` for file-based or remote prompt registries later
- keep deterministic business rules in Python hooks instead of prompts

Once this pattern feels comfortable, scaling up is mostly a matter of composing more typed steps.


In [ ]:
print(result.model_dump_json(indent=2))
